# Sistemas Dinámicos: Hegselmann-Krause
**Módulo 4 — Sistemas Dinámicos y Solución Numérica de EDOs**

**Programación Científica 2026-1 · Universidad Nacional de Colombia**  
mbastidaso@unal.edu.co

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

Este notebook define y visualiza un modelo de dinámica social:

1. **Hegselmann-Krause (H-K)**: polarización de opiniones bajo un umbral de confianza $\varepsilon$.


In [1]:
# ── Instalación de LaTeX para renderizado tipográfico (descomentar en Colab) ──
#!sudo apt-get update -qq
#!sudo apt-get install -qq texlive-latex-extra texlive-fonts-recommended dvipng cm-super

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

# ── Configuración global de estilo matplotlib ────────────────────────────────
# Renderiza etiquetas con LaTeX y fuente serif (Computer Modern), igual que
# los slides de la clase. Mejora la legibilidad en figuras publicables.
plt.rcParams.update({
    "text.usetex"        : True,
    "font.family"        : "serif",
    "font.serif"         : ["Computer Modern Roman"],
    "font.size"          : 14,

    # Ejes y ticks
    "axes.labelsize"     : 16,
    "axes.titlesize"     : 18,
    "xtick.labelsize"    : 12,
    "ytick.labelsize"    : 12,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.direction"    : "in",
    "ytick.direction"    : "in",

    # Grilla
    "grid.color"         : "gray",
    "grid.linewidth"     : 0.3,
    "grid.alpha"         : 0.3,
    "grid.linestyle"     : "--",

    # Estética general
    "figure.dpi"         : 120,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "savefig.bbox"       : "tight",
    "savefig.dpi"        : 300,
})
print("Configuración OK")


Configuración OK


---
## Parte 1 — Modelo de Hegselmann-Krause (H-K)

El modelo H-K captura cómo los agentes actualizan sus opiniones
$x_i(t) \in [-1, 1]$ interactuando **solo** con quienes tienen opiniones cercanas:

$$
\dot{x}_i(t) =
\frac{1}{|\mathcal{N}_i(t)|}
\sum_{j \in \mathcal{N}_i(t)} \!\bigl(x_j(t) - x_i(t)\bigr),
\qquad
\mathcal{N}_i(t) = \{j : |x_j(t) - x_i(t)| < \varepsilon\}.
$$

El umbral $\varepsilon > 0$ modela el **radio de confianza**:
si la diferencia de opiniones supera $\varepsilon$, el agente $i$ ignora al agente $j$.


### 1.1 Campo vectorial para $N=2$ agentes

Con dos agentes el espacio de estados es $[-1,1]^2$.
El campo vectorial $(\dot{x}_1, \dot{x}_2)$ muestra cómo evoluciona
el par de opiniones en cada punto del plano.

La **banda azul** señala la región de interacción $|x_2 - x_1| < \varepsilon$.
Fuera de ella el campo es cero: no hay cambio de opinión.


In [2]:
# ── Campo vectorial H-K para N=2 agentes ────────────────────────────────────
# Se visualiza para tres umbrales representativos:
#   ε = 0.25 → muy restrictivo: fragmentación total
#   ε = 0.55 → intermedio: posible polarización en dos clusters
#   ε = 0.90 → permisivo: consenso global

eps_vals = [0.25, 0.55, 0.90]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, eps in zip(axes, eps_vals):
    # Malla de puntos en el espacio de estados [−1, 1]²
    x1, x2 = np.meshgrid(np.linspace(-1, 1, 18), np.linspace(-1, 1, 18))

    # Diferencia de opiniones entre los dos agentes
    diff = x2 - x1

    # Máscara: True donde los agentes están dentro del radio de confianza
    inside = np.abs(diff) < eps

    # Campo vectorial: cada agente se mueve hacia el otro si están dentro del radio
    # Fuera del radio: no hay interacción (campo = 0)
    dx1 = np.where(inside,  diff, 0.0)   # ẋ₁ = (x₂ − x₁) si |diff| < ε, else 0
    dx2 = np.where(inside, -diff, 0.0)   # ẋ₂ = (x₁ − x₂) si |diff| < ε, else 0

    # Magnitud del campo para el colormap
    mag = np.sqrt(dx1**2 + dx2**2)
    nf  = np.where(mag > 0, mag, 1.0)    # denominador seguro (evitar 0/0)

    # Quiver: flechas normalizadas coloreadas por magnitud
    qv = ax.quiver(x1, x2, dx1/nf, dx2/nf, mag,
                   cmap="plasma", scale=22, width=0.004, pivot="mid",
                   clim=[0, 2*eps])
    plt.colorbar(qv, ax=ax, label=r"$|f(x)|$", fraction=0.046, pad=0.04)

    # Diagonal x₁ = x₂: todo punto sobre esta recta es un estado de consenso
    ax.plot([-1, 1], [-1, 1], "k--", lw=1.2, alpha=0.5, label=r"$x_1 = x_2$")

    # Banda de confianza: zona donde los dos agentes interactúan
    ax.fill_between([-1, 1], [-1 - eps, 1 - eps], [-1 + eps, 1 + eps],
                    alpha=0.10, color="steelblue",
                    label=r"$|x_2 - x_1| < \varepsilon$")

    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.set_xlabel(r"$x_1$ (agente 1)")
    ax.set_ylabel(r"$x_2$ (agente 2)")
    ax.set_title(fr"$\varepsilon = {eps}$")
    ax.set_aspect("equal")
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(True)

plt.suptitle(r"Campo vectorial H-K con $N=2$ agentes")
plt.tight_layout()
plt.show()


RuntimeError: Failed to process string with tex because latex could not be found

Error in callback <function _draw_all_if_interactive at 0x788aadf69800> (for post_execute):


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 1800x600 with 6 Axes>

## ¿Cómo resolverias o harías predicciones con este modelo?

### **Lluvia de ideas:** Hagamos entre todos un promt para que un LLM nos ayude a resolver este modelo. ¿Cuál es la pregunta ideal?